# 🔊 Laboratorio 3: Espectrogramas y Redes Convolucionales (CNN)
## Caso de Estudio: Clasificación de Audio — del Waveform 1D a la Imagen Tiempo-Frecuencia

**Pontificia Universidad Javeriana Cali — Curso de Deep Learning**  
**Tutor:** Jan Polanco Velasco  
**Semestre:** 2026-2

---

### 🎯 Objetivo del Laboratorio

En los Labs 1 y 2 trabajamos con **series temporales 1D** (manchas solares) usando MLPs autorregresivos y RNNs. En este laboratorio cambiamos de dominio: **audio**. Una forma de onda es una señal 1D de alta frecuencia (16 000 muestras por segundo) donde ni la ventana plana ni la recurrencia son la herramienta ideal. La solución estándar es convertir el audio en una **imagen tiempo-frecuencia** —el **espectrograma**— y clasificarla con una **Red Neuronal Convolucional (CNN 2D)**.

El material adapta y moderniza el script `Slides/pages/clase_3_–_espectrogramas.py` (STFT + `Conv2D` + `softmax` de 3 clases): el dataset privado de `.wav` se reemplaza por **audio sintético reproducible** (3 clases tonales), y la cabeza `Flatten` original —que genera ~780 k parámetros y sobreajusta con pocos datos— se sustituye por **`GlobalAveragePooling2D + Dropout` (~23 k parámetros)**, en línea con el tema de la Sesión 3 (diagnóstico y regularización).

El laboratorio está estructurado para **practicar los 4 actos del deck**: diagnóstico clínico (curvas, overfitting con etiquetas aleatorias, EarlyStopping), regularización (Gradient Clipping y Dropout), procesamiento de señales (ventanas STFT, escala lineal vs. dB) y convolución/pooling (pooling a mano, fórmula de reducción, GAP vs. Flatten).

### 📋 Contenido

1. Marco teórico: STFT, espectrogramas y convolución 2D
2. Generación del dataset sintético de audio (3 clases)
3. Visualización de formas de onda
4. STFT → espectrograma (magnitud, ventanas deslizantes, escala lineal vs. dB, vista 3D)
5. Tensor 4D `[samples, freq, time, channels]` y etiquetas one-hot
6. Arquitectura CNN: Conv2D → Pooling → GAP → Dropout → softmax
7. Pooling a mano, fórmula de reducción y comparación **GAP vs. Flatten**
8. Entrenamiento con Early Stopping
9. Regularización en acción: **Gradient Clipping** y **Dropout (train vs. inference)**
10. Evaluación: accuracy, reporte de clasificación y matriz de confusión
11. Diagnóstico: curvas, visualización de filtros y experimento de **etiquetas aleatorias**
12. Conclusiones

> 💡 **Consejo:** ejecuta las celdas en orden. Las gráficas de Plotly son interactivas (haz *zoom*, pasa el cursor y usa la leyenda). El notebook está *clean* (sin salidas versionadas): ejecútalo para reproducir los resultados.

---
## 1. 📚 Marco Teórico: de la Onda 1D a la Imagen 2D

### 1.1 ¿Por qué un espectrograma?

Una forma de onda $x[n]$ de 1 s a 16 kHz tiene 16 000 muestras. Clasificarla directamente con una red densa exigiría 16 000 entradas por muestra y la red tendría que redescubrir la noción de *frecuencia*. La **Transformada de Fourier de Corto Tiempo (STFT)** expone esa estructura: corta la señal en ventanas solapadas y calcula la FFT de cada una.

$$X[m, k] = \sum_{n=0}^{N-1} x[n + mH]\, w[n]\, e^{-j 2\pi k n / N}, \qquad S[m,k] = |X[m,k]|$$

donde $N = 255$ es el largo de ventana (`frame_length`), $H = 128$ el salto (`frame_step`), $w[n]$ la ventana y $S$ el **espectrograma de magnitud** de forma $(124, 129)$: 124 tramas temporales × 129 bines de frecuencia. Un tono puro aparece como una **línea horizontal brillante**; un barrido (*chirp*) como una **diagonal**. El problema de audio se vuelve un problema de **visión**: detectar patrones locales en una imagen.

### 1.2 La Convolución 2D

Una capa `Conv2D` desliza $F$ filtros $3 \times 3$ sobre la imagen y responde a patrones locales (bordes, líneas, texturas), compartiendo pesos en toda la imagen:

$$(I * K)[i,j] = \sum_{u=-1}^{1} \sum_{v=-1}^{1} I[i+u,\, j+v]\, K[u,v]$$

| Operación | Efecto | Forma (este lab) |
|---|---|---|
| `Conv2D(16/32/64, 3, relu, same)` | Detecta motivos tiempo-frecuencia | $(124,129,1) \to (124,129,16) \to \dots$ |
| `MaxPooling2D(2)` | Invariancia local + reduce 4× el área | $(124,129) \to (62,64) \to (31,32)$ |
| `GlobalAveragePooling2D` | Un valor por mapa → vector de 64 | $(31,32,64) \to (64,)$ |
| `Dropout(0.3)` + `Dense(3, softmax)` | Regulariza y clasifica | $(64,) \to (3,)$ |

### 1.3 Conexión con la Sesión 3: diagnóstico y regularización

- **Diagnóstico:** las curvas `loss/val_loss` revelan *overfitting* (brecha train/val), *underfitting* (ambas altas) o buen ajuste. Las veremos en la Sección 7.
- **Regularización:** `Dropout` apaga neuronas al azar durante el entrenamiento; `EarlyStopping` restaura los mejores pesos; `Adam(clipnorm=1.0)` recorta el gradiente global (antídoto contra gradientes que explotan).
- **Mejora sobre el script original:** el script usaba `Flatten()` tras una sola reducción ($62 \times 65 \times 64 \approx 258\,000$ valores → capa densa de ~774 k parámetros para solo 12 ejemplos de entrenamiento). Aquí usamos **GAP**: 64 valores → 195 parámetros en la densa. Menos parámetros = menos sobreajuste + entrenamiento en segundos en CPU.

---
## 2. ⚙️ Configuración Inicial

In [ ]:
import time
import warnings

import numpy as np
import plotly.graph_objects as go
import tensorflow as tf
from plotly.subplots import make_subplots
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import (
    Conv2D,
    Dense,
    Dropout,
    GlobalAveragePooling2D,
    Input,
    MaxPooling2D,
)
from tensorflow.keras.models import Model, Sequential

warnings.filterwarnings("ignore")

# Reproducibilidad
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version: {tf.__version__}")
print(f"Keras version:      {tf.keras.__version__}")
print(f"GPU disponible:     {len(tf.config.list_physical_devices('GPU')) > 0}")

---
## 3. 🔊 Dataset Sintético de Audio (3 Clases)

El script original leía `.wav` privados (`audio_path.glob('*.wav')`, índices fijos 7–27). Para que el laboratorio sea **reproducible sin archivos externos**, sintetizamos 3 clases de 1 s a 16 kHz con fase aleatoria y ruido gaussiano leve ($\sigma = 0.05$), normalizadas a amplitud máxima 1:

| Clase | Nombre | Contenido | Firma en el espectrograma |
|---|---|---|---|
| 0 | Grave | Seno 220 Hz | Línea horizontal baja |
| 1 | Agudo | Seno 880 Hz | Línea horizontal alta |
| 2 | Barrido (*chirp*) | Seno con frecuencia 200 → 1500 Hz | Diagonal ascendente |

Generamos **60 ejemplos por clase para entrenamiento** (180 total, semilla 7) y **15 por clase para prueba** (45 total, semilla 123): conjuntos disjuntos como en el script original (4 train + 2 test por clase), pero con volumen suficiente para entrenar una CNN de forma estable en CPU.

In [ ]:
SAMPLE_RATE = 16000
DURATION = 1.0
N_SAMPLES = int(SAMPLE_RATE * DURATION)
TIME_AXIS = np.arange(N_SAMPLES) / SAMPLE_RATE
CLASS_NAMES = ["Grave 220 Hz", "Agudo 880 Hz", "Barrido 200→1500 Hz"]
N_TRAIN_PER_CLASS = 60
N_TEST_PER_CLASS = 15


def synthesize_waveform(class_id, rng):
    """Genera 1 s de audio mono para la clase dada (fase y ruido aleatorios)."""
    phase = rng.uniform(0, 2 * np.pi)
    if class_id == 0:
        wave = np.sin(2 * np.pi * 220 * TIME_AXIS + phase)
    elif class_id == 1:
        wave = np.sin(2 * np.pi * 880 * TIME_AXIS + phase)
    else:
        freqs = 200 + (1500 - 200) * TIME_AXIS / DURATION
        wave = np.sin(2 * np.pi * np.cumsum(freqs) / SAMPLE_RATE + phase)
    wave = wave + 0.05 * rng.standard_normal(TIME_AXIS.shape)
    return (wave / np.max(np.abs(wave))).astype("float32")


def build_waveform_set(n_per_class, seed):
    """Genera ondas y etiquetas enteras, mezcladas con semilla fija."""
    rng = np.random.default_rng(seed)
    waves, labels = [], []
    for class_id in range(3):
        for _ in range(n_per_class):
            waves.append(synthesize_waveform(class_id, rng))
            labels.append(class_id)
    waves = np.stack(waves).astype("float32")
    labels = np.array(labels, dtype="int32")
    order = rng.permutation(len(waves))
    return waves[order], labels[order]


train_waves, train_labels = build_waveform_set(N_TRAIN_PER_CLASS, seed=7)
test_waves, test_labels = build_waveform_set(N_TEST_PER_CLASS, seed=123)

print(f"Train: ondas {train_waves.shape} etiquetas {train_labels.shape}")
print(f"Test:  ondas {test_waves.shape} etiquetas {test_labels.shape}")
print(f"Balance train: {np.bincount(train_labels)} | Balance test: {np.bincount(test_labels)}")
print(f"Ejemplo: {train_waves.shape[1]} muestras = {DURATION} s a {SAMPLE_RATE} Hz")

### 3.1 Visualización de las formas de onda

Un ejemplo por clase (primeros 2 000 muestras ≈ 125 ms para apreciar los ciclos). Observa: el tono grave oscila lento, el agudo vibra rápido y el barrido acelera su oscilación.

In [ ]:
ZOOM = 2000
COLORS = ["#4f46e5", "#ef4444", "#10b981"]

fig = make_subplots(rows=3, cols=1, subplot_titles=[f"Clase {i}: {name}" for i, name in enumerate(CLASS_NAMES)], vertical_spacing=0.12)
for class_id in range(3):
    idx = np.where(train_labels == class_id)[0][0]
    fig.add_trace(
        go.Scatter(y=train_waves[idx, :ZOOM], mode="lines", name=CLASS_NAMES[class_id], line=dict(color=COLORS[class_id], width=1.2)),
        row=class_id + 1,
        col=1,
    )
fig.update_layout(height=600, title="Formas de onda (zoom 125 ms)", showlegend=False)
fig.update_xaxes(title_text="Muestras")
fig.update_yaxes(title_text="Amplitud")
fig.show()

---
## 4. 📊 De la Onda al Espectrograma (STFT)

Aplicamos `tf.signal.stft` con los mismos hiperparámetros del script original (`frame_length=255`, `frame_step=128`) y tomamos la magnitud. Con 16 000 muestras obtenemos $(16000-255)//128+1 = 124$ tramas × $(255//2+1) = 129$ bines = tensor $(124, 129)$.

In [ ]:
FRAME_LENGTH = 255
FRAME_STEP = 128


def waveform_to_spectrogram(wave):
    """STFT + magnitud. Entrada (16000,) → salida (124, 129)."""
    stft = tf.signal.stft(wave, frame_length=FRAME_LENGTH, frame_step=FRAME_STEP)
    return tf.abs(stft)


demo_wave = train_waves[0]
demo_spec = waveform_to_spectrogram(demo_wave)
print(f"Waveform shape:    {demo_wave.shape}")
print(f"Spectrogram shape: {tuple(demo_spec.shape)}  (tramas × bines de frecuencia)")

### 4.1 El espectrograma en 2D y 3D

Visualizamos en escala logarítmica ($\log(S + 10^{-6})$) para comprimir el rango dinámico —igual que el `np.log(spectrograma.T)` del script—. La vista 3D usa un submuestreo de factor 2 para mantenerla ligera.

In [ ]:
def plot_spectrogram(spec, title="Espectrograma (log)"):
    """Heatmap Tiempo × Frecuencia en escala logarítmica."""
    log_spec = np.log(spec.numpy().T + 1e-6)
    fig = go.Figure(data=go.Heatmap(z=log_spec, colorscale="Viridis"))
    fig.update_layout(title=title, xaxis_title="Trama temporal", yaxis_title="Bín de frecuencia")
    return fig


example_specs = {}
for class_id in range(3):
    idx = np.where(train_labels == class_id)[0][0]
    example_specs[class_id] = waveform_to_spectrogram(train_waves[idx])

plot_spectrogram(example_specs[0], title="Clase 0 (Grave): línea horizontal baja").show()
plot_spectrogram(example_specs[2], title="Clase 2 (Barrido): diagonal ascendente").show()

# Superficie 3D submuestreada (ligera)
mag = example_specs[2].numpy()[::2, ::2]
fig3d = go.Figure(data=[go.Surface(z=np.log(mag + 1e-6), colorscale="Viridis")])
fig3d.update_layout(title="Espectrograma 3D — Barrido (log, submuestreado ×2)", autosize=True)
fig3d.show()

### 4.2 Ventanas Deslizantes y Escala Lineal vs. dB

La STFT **trocea** la señal con ventanas solapadas (`frame_length=255`, `frame_step=128`): de ahí que 1 s se convierta en 124 tramas. Además, como la magnitud abarca varios órdenes de magnitud, se grafica en **escala logarítmica (dB)** para revelar los formantes débiles —igual que el conmutador *lineal/dB* del componente `SpectrogramHeatmap` del deck.

In [ ]:
# (a) Ventanas deslizantes de la STFT sobre una forma de onda
demo = train_waves[0]
fig = go.Figure()
fig.add_trace(go.Scatter(x=np.arange(1200), y=demo[:1200], mode="lines",
                         name="x(t)", line=dict(color="#475569", width=1)))
for k, start in enumerate([0, 128, 256]):
    fig.add_vrect(x0=start, x1=start + FRAME_LENGTH, fillcolor="#6366f1",
                  opacity=0.15, line_width=0,
                  annotation_text=f"w{k+1}", annotation_position="top left")
fig.update_layout(title="Ventanas STFT (frame_length=255, frame_step=128)",
                  xaxis_title="Muestra", yaxis_title="Amplitud", height=320)
fig.show()

# (b) La misma trama en escala lineal vs logarítmica (dB)
spec = example_specs[2].numpy()   # espectrograma del barrido ya calculado en 4.1
fig = make_subplots(rows=1, cols=2, subplot_titles=["Escala lineal", "Escala logarítmica (dB)"])
fig.add_trace(go.Heatmap(z=spec.T, colorscale="Viridis", showscale=False), row=1, col=1)
fig.add_trace(go.Heatmap(z=np.log(spec.T + 1e-6), colorscale="Viridis", showscale=False), row=1, col=2)
fig.update_xaxes(title_text="Trama temporal")
fig.update_yaxes(title_text="Bín de frecuencia", col=1)
fig.update_layout(title="¿Por qué graficamos el espectrograma en dB?", height=400)
fig.show()

---
## 5. 🧱 Tensor 4D y Etiquetas One-Hot

Las capas `Conv2D` de Keras exigen un tensor **4D** `[samples, height, width, channels]`: aquí $(180, 124, 129, 1)$. Las etiquetas enteras se convierten a **one-hot** para `categorical_crossentropy`, igual que el `keras.utils.to_categorical` del script.

In [ ]:
def waveforms_to_tensor(waves):
    specs = np.stack([waveform_to_spectrogram(w).numpy() for w in waves]).astype("float32")
    return np.expand_dims(specs, axis=-1)


X_train = waveforms_to_tensor(train_waves)
X_test = waveforms_to_tensor(test_waves)
Y_train = tf.keras.utils.to_categorical(train_labels, num_classes=3)
Y_test = tf.keras.utils.to_categorical(test_labels, num_classes=3)

print(f"X_train {X_train.shape}  Y_train {Y_train.shape}")
print(f"X_test  {X_test.shape}  Y_test  {Y_test.shape}")
print(f"Ejemplo one-hot (clase {train_labels[0]}): {Y_train[0]}")

# Mosaico 3×3: un espectrograma por clase y columna
fig = make_subplots(rows=3, cols=3, subplot_titles=[f"{CLASS_NAMES[c]} #{k+1}" for c in range(3) for k in range(3)], horizontal_spacing=0.05, vertical_spacing=0.1)
for class_id in range(3):
    idxs = np.where(train_labels == class_id)[0][:3]
    for k, idx in enumerate(idxs):
        fig.add_trace(go.Heatmap(z=np.log(X_train[idx, :, :, 0].T + 1e-6), colorscale="Viridis", showscale=False), row=class_id + 1, col=k + 1)
fig.update_layout(height=700, title="Mosaico de espectrogramas (log) — filas = clases")
fig.show()

---
## 6. 🧠 Arquitectura CNN

Tres bloques `Conv2D + MaxPooling2D` con filtros crecientes (16 → 32 → 64), seguidos de `GlobalAveragePooling2D`, `Dropout(0.3)` y `Dense(3, softmax)`. El optimizador `Adam(clipnorm=1.0)` recorta la norma global del gradiente (tema de la Sesión 3: gradientes que explotan).

| Bloque | Capa | Salida | Parámetros |
|---|---|---|---|
| Entrada | `Input(124, 129, 1)` | $(124,129,1)$ | 0 |
| 1 | `Conv2D(16, 3, relu, same)` + `MaxPool(2)` | $(62,64,16)$ | 160 |
| 2 | `Conv2D(32, 3, relu, same)` + `MaxPool(2)` | $(31,32,32)$ | 4 640 |
| 3 | `Conv2D(64, 3, relu, same)` | $(31,32,64)$ | 18 496 |
| Cabeza | `GAP` → `Dropout(0.3)` → `Dense(3, softmax)` | $(3,)$ | 195 |
| **Total** | | | **23 491** |

> 💡 El script original apilaba `Conv2D(8/16/32/64)` + un solo `MaxPool` + `Flatten` (~774 k parámetros en la densa). Con 12 ejemplos, esa cabeza memoriza el ruido. GAP colapsa cada mapa a su promedio: la red vota por *presencia* del patrón, no por su posición exacta —invariancia espacial gratis y 33× menos parámetros.

In [ ]:
model = Sequential(
    [
        Input(shape=(124, 129, 1), name="espectrograma"),
        Conv2D(16, 3, activation="relu", padding="same", name="conv1"),
        MaxPooling2D(pool_size=2, name="pool1"),
        Conv2D(32, 3, activation="relu", padding="same", name="conv2"),
        MaxPooling2D(pool_size=2, name="pool2"),
        Conv2D(64, 3, activation="relu", padding="same", name="conv3"),
        GlobalAveragePooling2D(name="gap"),
        Dropout(0.3, name="dropout"),
        Dense(3, activation="softmax", name="clasificador"),
    ],
    name="CNN_espectrogramas",
)

model.compile(optimizer=tf.keras.optimizers.Adam(clipnorm=1.0), loss="categorical_crossentropy", metrics=["accuracy"])
model.summary()
print(f"Parámetros totales: {model.count_params():,}")

### 6.1 Pooling a Mano y la Fórmula de Reducción

El **pooling** reduce el tamaño espacial. Su dimensionalidad de salida es:

$$O = \left\lfloor \frac{N - f}{s} \right\rfloor + 1$$

donde $N$ es el tamaño de entrada, $f$ el tamaño del filtro y $s$ el *stride*. Implementemos **Max** y **Average Pooling** a mano sobre la matriz del componente `PoolingSimulator` y verifiquemos la fórmula con los tensores reales del modelo.

In [ ]:
def max_pool2d(x, f=2, s=2):
    '''Max Pooling 2D manual.'''
    H, W = x.shape
    oh, ow = (H - f) // s + 1, (W - f) // s + 1
    out = np.zeros((oh, ow))
    for i in range(oh):
        for j in range(ow):
            out[i, j] = x[i * s:i * s + f, j * s:j * s + f].max()
    return out


def avg_pool2d(x, f=2, s=2):
    '''Average Pooling 2D manual.'''
    H, W = x.shape
    oh, ow = (H - f) // s + 1, (W - f) // s + 1
    out = np.zeros((oh, ow))
    for i in range(oh):
        for j in range(ow):
            out[i, j] = x[i * s:i * s + f, j * s:j * s + f].mean()
    return out


# La misma matriz 4x4 del componente PoolingSimulator del deck
M = np.array([[5, 6, 6, 5], [9, 1, 2, 8], [4, 7, 1, 8], [3, 2, 2, 7]], dtype=float)
M_max, M_avg = max_pool2d(M), avg_pool2d(M)
print("Max Pooling:\n", M_max, "\nAverage Pooling:\n", np.round(M_avg, 2))

fig = make_subplots(rows=1, cols=3, subplot_titles=["Entrada (4×4)", "Max Pooling (2×2)", "Average Pooling (2×2)"])
fig.add_trace(go.Heatmap(z=M, colorscale="Blues", text=M.astype(int), texttemplate="%{text}", showscale=False), row=1, col=1)
fig.add_trace(go.Heatmap(z=M_max, colorscale="Reds", text=M_max.astype(int), texttemplate="%{text}", showscale=False), row=1, col=2)
fig.add_trace(go.Heatmap(z=M_avg, colorscale="Greens", text=np.round(M_avg, 2), texttemplate="%{text}", showscale=False), row=1, col=3)
fig.update_layout(title="Max vs. Average Pooling (f = 2, s = 2)", height=360)
fig.show()

# Verificación con la fórmula: 124 y 129 con f=2, s=2
print(f"O(124) = floor((124 - 2)/2) + 1 = {(124 - 2) // 2 + 1}")
print(f"O(129) = floor((129 - 2)/2) + 1 = {(129 - 2) // 2 + 1}")
print("Forma real tras pool1:", model.get_layer('pool1').output.shape)
print("Forma real tras pool2:", model.get_layer('pool2').output.shape)

### 6.2 ¿Por qué `GlobalAveragePooling2D` y no `Flatten`?

`Flatten` apila **todas** las posiciones espaciales y las conecta a una capa densa enorme. `GlobalAveragePooling2D` promedia cada mapa de características y produce un vector del tamaño del número de filtros. Comparemos el número de parámetros: es la diferencia entre memorizar y generalizar cuando hay pocos datos.

In [ ]:
tf.keras.utils.set_random_seed(42)
model_flat = Sequential(
    [
        Input(shape=(124, 129, 1)),
        Conv2D(16, 3, activation="relu", padding="same"),
        MaxPooling2D(pool_size=2),
        Conv2D(32, 3, activation="relu", padding="same"),
        MaxPooling2D(pool_size=2),
        Conv2D(64, 3, activation="relu", padding="same"),
        tf.keras.layers.Flatten(),
        Dense(3, activation="softmax"),
    ],
    name="CNN_flatten",
)

print(f"Cabeza Flatten : {model_flat.count_params():>9,} parámetros")
print(f"Cabeza GAP     : {model.count_params():>9,} parámetros")
print(f"Reducción      : {model_flat.count_params() / model.count_params():.1f}× menos con GAP")

---
## 7. 🏋️ Entrenamiento

`EarlyStopping(patience=8, restore_best_weights=True)` detiene el entrenamiento cuando `val_loss` deja de mejorar y restaura los mejores pesos —la receta contra el *overfitting* de la Sesión 3. Se reserva 20 % del train como validación (los datos ya vienen mezclados con semilla fija).

In [ ]:
early_stop = EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True, verbose=1)

start = time.perf_counter()
history = model.fit(X_train, Y_train, validation_split=0.2, epochs=40, batch_size=16, callbacks=[early_stop], verbose=2)
elapsed = time.perf_counter() - start
print(f"\nEntrenado en {elapsed:.1f} s · {len(history.history['loss'])} épocas")

In [ ]:
def plot_learning_curves(history):
    """Pérdida (cross-entropy) y accuracy en train/validación."""
    epochs = list(range(1, len(history.history["loss"]) + 1))
    fig = make_subplots(rows=1, cols=2, subplot_titles=["Función de costo (cross-entropy)", "Accuracy"])
    fig.add_trace(go.Scatter(x=epochs, y=history.history["loss"], mode="lines", name="train", line=dict(color="#4f46e5", width=2.2)), row=1, col=1)
    fig.add_trace(go.Scatter(x=epochs, y=history.history["val_loss"], mode="lines", name="validación", line=dict(color="#ef4444", width=2.2, dash="dash")), row=1, col=1)
    fig.add_trace(go.Scatter(x=epochs, y=history.history["accuracy"], mode="lines", name="train", line=dict(color="#4f46e5", width=2.2), showlegend=False), row=1, col=2)
    fig.add_trace(go.Scatter(x=epochs, y=history.history["val_accuracy"], mode="lines", name="validación", line=dict(color="#ef4444", width=2.2, dash="dash"), showlegend=False), row=1, col=2)
    fig.update_layout(height=380, title="Curvas de aprendizaje — diagnóstico de overfitting")
    fig.update_xaxes(title_text="Época")
    fig.show()


plot_learning_curves(history)

### 7.1 Gradient Clipping en Acción

El deck mostró que los gradientes pueden **explotar**. Lo reproducimos: mismo modelo y mismos datos, con **SGD** y un *learning rate* alto, **sin** y **con** `clipnorm`. Sin recorte, la pérdida oscila y el modelo no aprende; con el recorte de norma, el entrenamiento se estabiliza y converge.

In [ ]:
def build_cnn_sgd(clipnorm):
    tf.keras.utils.set_random_seed(0)
    m = Sequential(
        [
            Input(shape=(124, 129, 1)),
            Conv2D(16, 3, activation="relu", padding="same"),
            MaxPooling2D(pool_size=2),
            Conv2D(32, 3, activation="relu", padding="same"),
            MaxPooling2D(pool_size=2),
            Conv2D(64, 3, activation="relu", padding="same"),
            GlobalAveragePooling2D(),
            Dropout(0.3),
            Dense(3, activation="softmax"),
        ]
    )
    m.compile(
        tf.keras.optimizers.SGD(learning_rate=0.5, clipnorm=clipnorm),
        "categorical_crossentropy",
        metrics=["accuracy"],
    )
    return m


hist_noclip = build_cnn_sgd(None).fit(X_train, Y_train, validation_data=(X_test, Y_test), epochs=25, batch_size=16, verbose=0)
hist_clip = build_cnn_sgd(1.0).fit(X_train, Y_train, validation_data=(X_test, Y_test), epochs=25, batch_size=16, verbose=0)

print(f"Sin clipping — val_acc: {hist_noclip.history['val_accuracy'][-1]:.3f} · loss final: {hist_noclip.history['loss'][-1]:.3f}")
print(f"Con clipping — val_acc: {hist_clip.history['val_accuracy'][-1]:.3f} · loss final: {hist_clip.history['loss'][-1]:.3f}")

fig = make_subplots(rows=1, cols=2, subplot_titles=["Sin Gradient Clipping", "Con clipnorm = 1.0"])
for h, col in [(hist_noclip, 1), (hist_clip, 2)]:
    ep = list(range(1, len(h.history["loss"]) + 1))
    fig.add_trace(go.Scatter(x=ep, y=h.history["loss"], name="train", line=dict(color="#4f46e5", width=2)), row=1, col=col)
    fig.add_trace(go.Scatter(x=ep, y=h.history["val_loss"], name="validación", line=dict(color="#ef4444", width=2, dash="dash")), row=1, col=col)
fig.update_layout(height=400, title="Gradient Clipping: estabilidad del entrenamiento (SGD, lr = 0.5)")
fig.update_xaxes(title_text="Época")
fig.update_yaxes(title_text="Pérdida (cross-entropy)", col=1)
fig.show()

### 7.2 Dropout: Entrenamiento vs. Inferencia

El Dropout es **estocástico solo durante el entrenamiento**: en inferencia todas las neuronas están activas (dropout invertido). Lo comprobamos sobre el modelo ya entrenado, observando las activaciones de su capa `dropout`: con `training=True` varían y aparecen ceros; con `training=False` son idénticas.

In [ ]:
probe = Model(inputs=model.inputs, outputs=model.get_layer("dropout").output)
x_probe = X_test[:1]

train_runs = np.stack([probe(x_probe, training=True).numpy()[0] for _ in range(4)])
infer_runs = np.stack([probe(x_probe, training=False).numpy()[0] for _ in range(4)])

print("Entrenamiento (training=True) — primeras 6 activaciones:")
for r in train_runs:
    print("  ", np.round(r[:6], 3), "| ceros:", int((r == 0).sum()), "/", r.size)

print("Inferencia (training=False) — primeras 6 activaciones:")
for r in infer_runs:
    print("  ", np.round(r[:6], 3), "| ceros:", int((r == 0).sum()), "/", r.size)

print("¿Inferencia idéntica en las 4 pasadas?", bool(np.allclose(infer_runs[0], infer_runs[1])))

---
## 8. 🎯 Evaluación en el Conjunto de Prueba

El conjunto de prueba (45 ejemplos, semilla distinta) nunca se usó para entrenar ni para early stopping: mide generalización real. Reportamos `accuracy`, `classification_report` (precisión/recall/F1 por clase) y la **matriz de confusión** como heatmap de Plotly (el script usaba seaborn; aquí evitamos esa dependencia).

In [ ]:
test_loss, test_acc = model.evaluate(X_test, Y_test, verbose=0)
print(f"Test loss: {test_loss:.4f} · Test accuracy: {test_acc:.4f}")

y_pred_prob = model.predict(X_test, verbose=0)
y_pred = np.argmax(y_pred_prob, axis=1)

print(classification_report(test_labels, y_pred, target_names=CLASS_NAMES, zero_division=0))
cm = confusion_matrix(test_labels, y_pred)
print(cm)

fig = go.Figure(data=go.Heatmap(z=cm, x=CLASS_NAMES, y=CLASS_NAMES, colorscale="Blues", showscale=True))
for i in range(3):
    for j in range(3):
        fig.add_annotation(x=CLASS_NAMES[j], y=CLASS_NAMES[i], text=str(cm[i, j]), showarrow=False, font=dict(color="white" if cm[i, j] > cm.max() / 2 else "black", size=16))
fig.update_layout(title=f"Matriz de confusión (test, acc={test_acc:.2f})", xaxis_title="Predicción", yaxis_title="Real")
fig.show()

# Mosaico de predicciones: 2 aciertos + 2 errores (o aciertos si el modelo es perfecto)
correct = np.where(y_pred == test_labels)[0]
errors = np.where(y_pred != test_labels)[0]
show_idx = list(correct[:2]) + list(errors[:2] if len(errors) else correct[2:4])
fig = make_subplots(rows=2, cols=2, subplot_titles=[f"Real: {CLASS_NAMES[test_labels[i]]} → Pred: {CLASS_NAMES[y_pred[i]]} ({y_pred_prob[i, y_pred[i]]:.2f})" for i in show_idx])
for k, i in enumerate(show_idx):
    fig.add_trace(go.Heatmap(z=np.log(X_test[i, :, :, 0].T + 1e-6), colorscale="Viridis", showscale=False), row=k // 2 + 1, col=k % 2 + 1)
fig.update_layout(height=600, title="Ejemplos de predicción (espectrogramas log)")
fig.show()

---
## 9. 🔍 Diagnóstico: ¿Overfitting? ¿Qué Ven los Filtros?

**Lectura de las curvas (Sección 7):** si `val_loss` baja junto a `loss` → buen ajuste; si `val_loss` sube mientras `loss` sigue bajando → *overfitting* (aquí lo contienen GAP + Dropout + EarlyStopping). Con 180 ejemplos sintéticos bien separados, el modelo converge a `val_accuracy ≈ 1.0` en apenas unas pocas épocas (≈ 5): es un problema *demasiado fácil*, de modo que aquí la regularización sirve sobre todo como red de seguridad.

Para abrir la caja negra, visualizamos los **mapas de activación** de `conv1` (16 filtros) sobre un barrido: cada mapa resalta qué parte del espectrograma excita a cada filtro.

In [ ]:
probe_idx = np.where(test_labels == 2)[0][0]
activation_model = Model(inputs=model.inputs, outputs=model.get_layer("conv1").output)
feature_maps = activation_model.predict(X_test[probe_idx : probe_idx + 1], verbose=0)[0]
print(f"Mapas de conv1: {feature_maps.shape} (alto × ancho × filtros)")

fig = make_subplots(rows=2, cols=4, subplot_titles=[f"Filtro {f}" for f in range(8)])
for f in range(8):
    fig.add_trace(go.Heatmap(z=feature_maps[:, :, f], colorscale="Viridis", showscale=False), row=f // 4 + 1, col=f % 4 + 1)
fig.update_layout(height=500, title="Activaciones de conv1 ante un barrido (8 de 16 filtros)")
fig.show()

---
## 10. 🔬 Diagnóstico: ¿Puede la Red Memorizar Ruido?

En el ejercicio anterior el problema era **demasiado separable**, así que train y validación bajaban juntas (buen ajuste). Para **exponer el sobreajuste de forma controlada** usamos el experimento clásico de *etiquetas aleatorias* (Zhang et al., 2017): mantenemos los mismos 24 espectrogramas, pero asignamos **etiquetas al azar**.

La tarea es imposible de aprender (no existe relación entrada-etiqueta), pero una red con capacidad suficiente puede **memorizar** el conjunto de entrenamiento. Si la pérdida de entrenamiento colapsa a 0 mientras la de validación se estanca o sube, estamos viendo **sobreajuste puro** (memorización sin generalización): el diagnóstico central de la Sesión 3.

| Escenario | `train_loss` | `val_loss` | Diagnóstico |
|---|---|---|---|
| Etiquetas reales | ↓ 0 | ↓ 0 | Buen ajuste |
| Subajuste (poca capacidad/épocas) | alta | alta | Subajuste |
| Etiquetas aleatorias | ↓ 0 | se estanca / sube | **Sobreajuste** |

> 🔎 **EarlyStopping:** cuando `val_loss` repunta, detenemos el entrenamiento y restauramos los pesos de la mejor época (lo demostramos abajo).

> 🛡️ **Antídotos:** más datos, `Dropout`, `EarlyStopping`, reducción de capacidad (aquí la cabeza `Flatten` de 190 k parámetros es la que memoriza; `GlobalAveragePooling2D` la reduciría a ~200).

In [ ]:
# Subconjunto pequeño: 8 ejemplos por clase (24 en total)
SMALL_PER_CLASS = 8
subset_idx = np.concatenate([np.where(train_labels == c)[0][:SMALL_PER_CLASS] for c in range(3)])
X_small = X_train[subset_idx]
Y_small_true = Y_train[subset_idx]

# Etiquetas aleatorias: tarea imposible, útil para exponer la memorización
rng = np.random.default_rng(0)
random_labels = rng.integers(0, 3, size=len(subset_idx))
Y_small_random = tf.keras.utils.to_categorical(random_labels, 3)
print(f"Subconjunto: {X_small.shape} · etiquetas aleatorias: {random_labels}")


def build_high_capacity():
    '''Cabeza de alta capacidad (Flatten + Dense): la del script original.'''
    tf.keras.utils.set_random_seed(42)
    m = Sequential(
        [
            Input(shape=(124, 129, 1)),
            Conv2D(16, 3, activation="relu", padding="same"),
            MaxPooling2D(pool_size=2),
            tf.keras.layers.Flatten(),
            Dense(3, activation="softmax"),
        ]
    )
    m.compile(
        optimizer=tf.keras.optimizers.Adam(clipnorm=1.0),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )
    return m


def train_on(labels, epochs=80):
    m = build_high_capacity()
    return m.fit(X_small, labels, validation_data=(X_test, Y_test),
                 epochs=epochs, batch_size=8, verbose=0)


hist_true = train_on(Y_small_true)
hist_random = train_on(Y_small_random)

print(f"Etiquetas reales     — train_loss: {hist_true.history['loss'][-1]:.4f} · val_loss: {hist_true.history['val_loss'][-1]:.4f} · val_acc: {hist_true.history['val_accuracy'][-1]:.3f}")
print(f"Etiquetas aleatorias — train_loss: {hist_random.history['loss'][-1]:.4f} · val_loss: {hist_random.history['val_loss'][-1]:.4f} · val_acc: {hist_random.history['val_accuracy'][-1]:.3f}")

fig = make_subplots(rows=1, cols=2, subplot_titles=["Etiquetas reales (buen ajuste)", "Etiquetas aleatorias (sobreajuste)"])
for h, col in [(hist_true, 1), (hist_random, 2)]:
    ep = list(range(1, len(h.history["loss"]) + 1))
    fig.add_trace(go.Scatter(x=ep, y=h.history["loss"], name="train", line=dict(color="#4f46e5", width=2)), row=1, col=col)
    fig.add_trace(go.Scatter(x=ep, y=h.history["val_loss"], name="validación", line=dict(color="#ef4444", width=2, dash="dash")), row=1, col=col)

fig.update_layout(height=420, title="Diagnóstico: la red puede memorizar etiquetas aleatorias (sobreajuste)")
fig.update_xaxes(title_text="Época")
fig.update_yaxes(title_text="Pérdida (cross-entropy)", col=1)
fig.show()


# --- EarlyStopping sobre el mismo escenario (etiquetas aleatorias) ---
es = EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True, verbose=0)
m_es = build_high_capacity()
hist_es = m_es.fit(X_small, Y_small_random, validation_data=(X_test, Y_test),
                   epochs=80, batch_size=8, callbacks=[es], verbose=0)
best_ep = int(np.argmin(hist_es.history["val_loss"])) + 1
print(f"Sin EarlyStopping — val_loss final: {hist_random.history['val_loss'][-1]:.4f}")
print(f"Con EarlyStopping — detuvo en época {len(hist_es.history['loss'])}; restauró la mejor: val_loss={min(hist_es.history['val_loss']):.4f} (época {best_ep})")

fig_es = go.Figure(go.Bar(
    x=["Sin ES (final)", "Con ES (mejor)"],
    y=[hist_random.history["val_loss"][-1], min(hist_es.history["val_loss"])],
    marker_color=["#ef4444", "#10b981"],
    text=[f"{hist_random.history['val_loss'][-1]:.3f}", f"{min(hist_es.history['val_loss']):.3f}"],
    textposition="outside",
))
fig_es.update_layout(title="EarlyStopping conserva la pérdida de validación en su mejor punto",
                     yaxis_title="val_loss", height=320)
fig_es.show()

---
## 11. 🎓 Conclusiones

1. **El espectrograma convierte audio en visión:** la STFT expone la frecuencia como dimensión espacial y la CNN detecta motivos locales (líneas, diagonales) con pesos compartidos —mucho más eficiente que una red densa sobre 16 000 muestras.
2. **GAP + Dropout > Flatten con pocos datos:** la cabeza original (~774 k parámetros) memoriza; GAP (64 valores) + Dropout(0.3) generaliza con 23 k parámetros totales y entrena en segundos en CPU.
3. **Regularización verificada en el lab:** con SGD y `lr=0.5`, **sin** Gradient Clipping el modelo no aprende (val_acc ≈ 0.33) y **con** `clipnorm=1.0` converge; EarlyStopping restaura el mejor `val_loss` y Dropout solo se activa durante el entrenamiento (en inferencia es determinista).
4. **El sobreajuste se diagnostica en las curvas:** con etiquetas aleatorias, `train_loss → 0` mientras `val_loss` sube; la red memoriza incluso ruido. Más datos, `Dropout`, `EarlyStopping` y menor capacidad (`GAP` en vez de `Flatten`) son el antídoto.
5. **Convolución y pooling:** la fórmula $O = \lfloor (N-f)/s \rfloor + 1$ predice las formas reales (124 → 62 → 31); `GlobalAveragePooling2D` reduce ~9× los parámetros frente a `Flatten`, combatiendo el sobreajuste.
6. **Reproducibilidad sin archivos privados:** el audio sintético (fase y ruido aleatorios con semilla fija, splits disjuntos) reproduce fielmente el flujo del script original —`load_wav` → `preprocess` (STFT) → `Conv2D` → `softmax` → `classification_report` + matriz de confusión— sin depender de `.wav` externos.
7. **Puente al futuro (Sesión 4):** el mismo patrón —señal → representación 2D → CNN— aplica a imágenes, video y espectrogramas de voz para reconocimiento de comandos o *keyword spotting*.

---

**📚 Referencias:**

[1] Script del curso `Slides/pages/clase_3_–_espectrogramas.py` — STFT (`frame_length=255`, `frame_step=128`), CNN `Conv2D(8/16/32/64) + MaxPooling2D + Flatten + Dense(3, softmax)`, 12 train / 6 test, `classification_report` + matriz de confusión.  
[2] LeCun, Y. et al. (1998). *Gradient-Based Learning Applied to Document Recognition*. Proc. IEEE — convoluciones y pooling.  
[3] Srivastava, N. et al. (2014). *Dropout: A Simple Way to Prevent Neural Networks from Overfitting*. JMLR.  
[4] Prechelt, L. (1998). *Early Stopping — But When?* Neural Networks — criterio de parada por validación.